# Transform Data

Goal of this notebook is to convert all race data into a flat-file of the following schema:

- RaceId 
- DriverId
- TeamId
- Position (Starting)
- Starting Tyre Compound
- Starting Tyre freshness (boolean)  

Pit stop 1   
- A column to measure time taken to first pit stop  
- Lap at which first pit stop occurred
- Time taken for first pit stop
- Compound after first pit stop
- Freshness of tyre after first pit stop  

Pit stop 2  
- A column to measure time taken to second pit stop
- Lap at which second pit stop occurred
- Time taken for second pit stop
- Compound after second pit stop
- Freshness of tyre after second pit stop 

Pit stop three
- A column to measure time taken to third pit stop (if it exists)
- Lap at which third pit stop occurred (if it exists)
- Time taken for third pit stop (if it exists)
- Compound after third pit stop (if it exists)
- Freshness of tyre after third pit stop (if it exists)

- Number of laps to change tires after rain begins
- Number of laps to change tires after rain ends
- Rainfall present on track
- Maximum duration of rainfall (if present)
- Maximum difference in weather temperature
- Gradient of weather temperature
- Safety car deployed (multiple columns storing the lap at which it occurred, maybe go up to 72)
- Pit stops during a safety car
- Matrix of [Angle, Distance] for given track

Target
- GridPosition (Finishing, this will be our target)

In [19]:
import fastf1
import numpy as np
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_PATH = Path.cwd().parent / "data"
fastf1.Cache.enable_cache(DATA_PATH)

In [3]:
race = fastf1.get_session(2021, "Imola", "R")
race.load()

core           INFO 	Loading data for Emilia Romagna Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 33 completed the race distance 00:01.003000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '4', '16', '55', '3', '10', '18', '31', '14', '

## Race Starting Grid and Results

In [3]:
race.results['Position'] # finishing 

33     1.0
44     2.0
4      3.0
16     4.0
55     5.0
3      6.0
10     7.0
18     8.0
31     9.0
14    10.0
11    11.0
22    12.0
7     13.0
99    14.0
5     15.0
47    16.0
9     17.0
77    18.0
63    19.0
6     20.0
Name: Position, dtype: float64

In [4]:
race.results['GridPosition'] # starting 

33     3.0
44     1.0
4      7.0
16     4.0
55    11.0
3      6.0
10     5.0
18    10.0
31     9.0
14    15.0
11     2.0
22    20.0
7     16.0
99    17.0
5      0.0
47    18.0
9     19.0
77     8.0
63    12.0
6     14.0
Name: GridPosition, dtype: float64

In [5]:
race.laps.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

In [6]:
race.laps[['Driver', 'DriverNumber', 'LapNumber', 'PitInTime', 'FreshTyre', 'TyreLife', 'Compound', 'TrackStatus']]

,Driver,DriverNumber,LapNumber,PitInTime,FreshTyre,TyreLife,Compound,TrackStatus
0,GAS,10,1.0,NaT,True,1.0,WET,124
1,GAS,10,2.0,NaT,True,2.0,WET,4
2,GAS,10,3.0,NaT,True,3.0,WET,4
3,GAS,10,4.0,NaT,True,4.0,WET,4
4,GAS,10,5.0,NaT,True,5.0,WET,4
...,...,...,...,...,...,...,...,...
1122,GIO,99,58.0,NaT,False,31.0,MEDIUM,1
1123,GIO,99,59.0,NaT,False,32.0,MEDIUM,1
1124,GIO,99,60.0,NaT,False,33.0,MEDIUM,1
1125,GIO,99,61.0,NaT,False,34.0,MEDIUM,12


## Weather

In [ ]:
weather_data = race.weather_data.copy(deep=True)
# weather_data.assign(rain_start=lambda x: (x.Rainfall.shift(-1) == False and  x.Rainfall == True ))

weather_data['rain_start'] = (weather_data['Rainfall'] == True) & (weather_data['Rainfall'].shift(1) == False)
weather_data['rain_stop'] = (weather_data['Rainfall'].shift(1) == True) & (weather_data['Rainfall'] == False)
weather_data['rain_start_time'] = weather_data['Time'].where(weather_data['rain_start']) 
weather_data['rain_stop_time'] = weather_data['Time'].where(weather_data['rain_stop'])

weather_data['rain_start_time'] = weather_data['rain_start_time'].dt.total_seconds().astype(float)
weather_data['rain_stop_time'] = weather_data['rain_stop_time'].dt.total_seconds().astype(float)

weather_data['rain_start_time'] = weather_data['rain_start_time'].ffill()
weather_data['rain_stop_time'] = weather_data['rain_stop_time'].ffill()

weather_data['Time'] = weather_data['Time'].dt.total_seconds().astype(float)


weather_data['max_temp'] = weather_data['TrackTemp'].max()
weather_data['min_temp'] = weather_data['TrackTemp'].min()
weather_data['avg_temp'] = weather_data['TrackTemp'].mean()
weather_data['std_temp'] = weather_data['TrackTemp'].std()



weather_data

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,rain_start,rain_stop,rain_start_time,rain_stop_time,max_temp,min_temp,avg_temp,std_temp
0,41.291,9.8,68.7,1011.7,True,18.1,61,0.2,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
1,101.287,9.7,70.0,1011.6,True,18.1,261,0.1,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
2,161.294,9.6,69.2,1011.6,True,18.0,243,0.4,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
3,221.289,9.4,68.3,1011.6,True,17.7,250,0.4,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
4,281.288,9.3,67.9,1011.7,True,17.6,259,0.5,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
5,341.289,9.3,69.1,1011.6,True,17.6,256,0.5,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
6,401.296,9.3,69.4,1011.6,True,17.5,248,0.4,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
7,461.289,9.2,70.6,1011.6,True,17.5,268,0.4,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
8,521.293,9.2,70.5,1011.6,True,17.5,214,0.4,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921
9,581.290,9.2,70.7,1011.7,True,17.5,237,0.2,False,False,NaN,NaN,18.6,13.8,16.827329,1.301921


In [ ]:
corners = race.get_circuit_info().corners[['Angle', 'Distance']]
corners.values

array([[ -97.8239287 ,  348.84381074],
       [ 111.88313377,  967.34205354],
       [ -30.15716703, 1029.34771621],
       [ 140.59195787, 1169.99862496],
       [-178.31172885, 1602.72018903],
       [ -21.43865197, 1704.16521124],
       [-171.05139602, 1987.36188568],
       [  90.97300951, 2396.17181181],
       [ -41.14815521, 2644.74609896],
       [-180.56795043, 2770.2173105 ],
       [-191.14447209, 3034.04691098],
       [ 115.09022179, 3133.4751165 ],
       [ -90.97901268, 3206.81813543],
       [  64.02888553, 3632.2803438 ],
       [ -96.25082234, 3674.66006957],
       [ -51.56989886, 4229.65969617],
       [  -7.57990847, 4425.27764484],
       [  86.24746694, 4566.39225556],
       [ -73.50537254, 4872.57949364]])

## Race Starting/Ending Positions

In [12]:
start_end_positions = pd.concat([race.results['Position'],race.results['GridPosition']], axis=1).rename(columns={'Position': 'target', 'GridPosition': 'starting_position'}).astype({'starting_position': 'int', 'target': 'int'})
start_end_positions


,target,starting_position
33,1,3
44,2,1
4,3,7
16,4,4
55,5,11
3,6,6
10,7,5
18,8,10
31,9,9
14,10,15


In [13]:
laps = race.laps[['Driver', 'DriverNumber', 'LapNumber', 'PitInTime', 'PitOutTime', 'FreshTyre', 'TyreLife', 'Compound', 'TrackStatus']]

In [14]:
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)


(laps := race.laps
    # calcualte pit stop duration
    .assign(pit_duration=lambda x: (x.PitOutTime.shift(-1) - x.PitInTime).dt.total_seconds())
    # the lap at which the pit stop was made
    .assign(pit_stop=lambda x: np.where(x.pit_duration > 0, x.LapNumber.astype(int), None))
    # the strategy after the pit stop
    .assign(strategy_after_pit=lambda x: np.where(x.shift(-1).Compound.isin(['SOFT']), 'SOFT', np.where(x.shift(-1).Compound.isin(['MEDIUM']), 'MEDIUM', np.where(x.shift(-1).Compound.isin(['HARD']), 'HARD', None))))
    # rename the compound column to strategy before the pit stop
    .rename(columns={'Compound': 'strategy_before_pit'})
    # filter only for laps with pit stops
    .query('pit_stop.notnull()')
    [
['Driver', 'DriverNumber', 'LapNumber', 'PitInTime', 'FreshTyre', 'TyreLife', 'TrackStatus', 'pit_duration', 'pit_stop', 'strategy_after_pit', 'strategy_before_pit']
    ]
)

laps

,Driver,DriverNumber,LapNumber,PitInTime,FreshTyre,TyreLife,TrackStatus,pit_duration,pit_stop,strategy_after_pit,strategy_before_pit
13,GAS,10,14.0,0 days 01:00:10.679000,True,14.0,1,31.053,14,None,WET
25,GAS,10,26.0,0 days 01:18:45.618000,True,12.0,1,32.291,26,MEDIUM,INTERMEDIATE
31,GAS,10,32.0,0 days 01:29:18.556000,True,6.0,45,1453.525,32,MEDIUM,MEDIUM
32,GAS,10,33.0,0 days 01:55:10.239000,False,6.0,51,56.077,33,MEDIUM,MEDIUM
90,PER,11,28.0,0 days 01:19:58.896000,True,28.0,1,44.676,28,MEDIUM,INTERMEDIATE
95,PER,11,33.0,0 days 01:29:11.663000,True,5.0,45,1581.172,33,SOFT,MEDIUM
153,ALO,14,28.0,0 days 01:21:18.812000,True,28.0,1,30.984,28,MEDIUM,INTERMEDIATE
157,ALO,14,32.0,0 days 01:28:37.450000,True,4.0,45,1488.331,32,MEDIUM,MEDIUM
158,ALO,14,33.0,0 days 01:55:05.161000,False,4.0,51,57.588,33,MEDIUM,MEDIUM
216,LEC,16,28.0,0 days 01:19:44.152000,True,28.0,1,30.958,28,MEDIUM,INTERMEDIATE


In [15]:
# get the number of pit stops made for each driver
pit_stop_counts = laps.groupby('Driver')['pit_stop'].nunique().reset_index().rename(columns={'pit_stop': 'num_pit_stops'})
pit_stop_counts


laps_pit_stops = pd.merge(left=laps, right=pit_stop_counts, on='Driver', how='left')
laps_pit_stops

,Driver,DriverNumber,LapNumber,PitInTime,FreshTyre,TyreLife,TrackStatus,pit_duration,pit_stop,strategy_after_pit,strategy_before_pit,num_pit_stops
0,GAS,10,14.0,0 days 01:00:10.679000,True,14.0,1,31.053,14,None,WET,4
1,GAS,10,26.0,0 days 01:18:45.618000,True,12.0,1,32.291,26,MEDIUM,INTERMEDIATE,4
2,GAS,10,32.0,0 days 01:29:18.556000,True,6.0,45,1453.525,32,MEDIUM,MEDIUM,4
3,GAS,10,33.0,0 days 01:55:10.239000,False,6.0,51,56.077,33,MEDIUM,MEDIUM,4
4,PER,11,28.0,0 days 01:19:58.896000,True,28.0,1,44.676,28,MEDIUM,INTERMEDIATE,2
5,PER,11,33.0,0 days 01:29:11.663000,True,5.0,45,1581.172,33,SOFT,MEDIUM,2
6,ALO,14,28.0,0 days 01:21:18.812000,True,28.0,1,30.984,28,MEDIUM,INTERMEDIATE,3
7,ALO,14,32.0,0 days 01:28:37.450000,True,4.0,45,1488.331,32,MEDIUM,MEDIUM,3
8,ALO,14,33.0,0 days 01:55:05.161000,False,4.0,51,57.588,33,MEDIUM,MEDIUM,3
9,LEC,16,28.0,0 days 01:19:44.152000,True,28.0,1,30.958,28,MEDIUM,INTERMEDIATE,2


In [16]:
# Flatten/pivot the above such that we have one row per driver, and columns for each pit stop (lap number, duration, strategy before and after the pit stop)
laps_pit_stops['stop_num'] = laps_pit_stops.groupby('Driver').cumcount() + 1
per_stop_cols = {
        "LapNumber": "lap",
        "pit_duration": "duration",
        "PitInTime": "intime",
        "FreshTyre": "fresh",
        "strategy_before_pit": "tyre_before",
        "strategy_after_pit": "tyre_after",
    }

pivoted = laps_pit_stops.groupby('Driver').first()[["DriverNumber", "num_pit_stops"]].reset_index()


for src_col, label in per_stop_cols.items():
    stop_pivot = (
            laps_pit_stops.pivot_table(index='Driver', columns="stop_num", values=src_col, aggfunc="first")
        )
    stop_pivot.columns = [f"pit_{int(n)}_{label}" for n in stop_pivot.columns]
    pivoted = pivoted.merge(stop_pivot.reset_index(), on='Driver', how="left")


laps_pit_stops_flat = pivoted.sort_values('DriverNumber').reset_index(drop=True)
laps_pit_stops_flat


,Driver,DriverNumber,num_pit_stops,pit_1_lap,pit_2_lap,pit_3_lap,pit_4_lap,pit_5_lap,pit_1_duration,pit_2_duration,pit_3_duration,pit_4_duration,pit_5_duration,pit_1_intime,pit_2_intime,pit_3_intime,pit_4_intime,pit_5_intime,pit_1_fresh,pit_2_fresh,pit_3_fresh,pit_4_fresh,pit_5_fresh,pit_1_tyre_before,pit_2_tyre_before,pit_3_tyre_before,pit_4_tyre_before,pit_5_tyre_before,pit_1_tyre_after,pit_2_tyre_after,pit_3_tyre_after,pit_4_tyre_after,pit_5_tyre_after
0,GAS,10,4,14.0,26.0,32.0,33.0,NaN,31.053,32.291,1453.525,56.077,NaN,0 days 01:00:10.679000,0 days 01:18:45.618000,0 days 01:29:18.556000,0 days 01:55:10.239000,NaT,True,True,True,False,NaN,WET,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN
1,PER,11,2,28.0,33.0,NaN,NaN,NaN,44.676,1581.172,NaN,NaN,NaN,0 days 01:19:58.896000,0 days 01:29:11.663000,NaT,NaT,NaT,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,SOFT,NaN,NaN,NaN
2,ALO,14,3,28.0,32.0,33.0,NaN,NaN,30.984,1488.331,57.588,NaN,NaN,0 days 01:21:18.812000,0 days 01:28:37.450000,0 days 01:55:05.161000,NaT,NaT,True,True,False,NaN,NaN,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN,NaN
3,LEC,16,2,28.0,33.0,NaN,NaN,NaN,30.958,1609.473,NaN,NaN,NaN,0 days 01:19:44.152000,0 days 01:28:28.896000,NaT,NaT,NaT,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,MEDIUM,NaN,NaN,NaN
4,STR,18,3,27.0,32.0,33.0,NaN,NaN,31.137,1492.073,65.406,NaN,NaN,0 days 01:19:16.421000,0 days 01:28:14.191000,0 days 01:54:37.429000,NaT,NaT,True,True,False,NaN,NaN,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN,NaN
5,TSU,22,3,25.0,32.0,33.0,NaN,NaN,30.696,1491.687,63.320,NaN,NaN,0 days 01:16:39.754000,0 days 01:28:24.596000,0 days 01:54:53.306000,NaT,NaT,True,True,True,NaN,NaN,INTERMEDIATE,MEDIUM,SOFT,NaN,NaN,MEDIUM,SOFT,SOFT,NaN,NaN
6,RIC,3,2,27.0,33.0,NaN,NaN,NaN,34.285,1575.137,NaN,NaN,NaN,0 days 01:18:57.991000,0 days 01:29:26.051000,NaT,NaT,NaT,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,SOFT,NaN,NaN,NaN
7,OCO,31,5,1.0,27.0,31.0,32.0,33.0,30.809,30.642,30.678,1478.975,56.736,0 days 00:35:16.950000,0 days 01:19:42.796000,0 days 01:26:23.677000,0 days 01:28:48.344000,0 days 01:55:07.285000,True,True,True,False,False,WET,INTERMEDIATE,MEDIUM,SOFT,MEDIUM,NaN,MEDIUM,SOFT,MEDIUM,MEDIUM
8,VER,33,2,27.0,33.0,NaN,NaN,NaN,30.543,1621.405,NaN,NaN,NaN,0 days 01:18:00.076000,0 days 01:28:12.328000,NaT,NaT,NaT,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,MEDIUM,NaN,NaN,NaN
9,NOR,4,2,28.0,33.0,NaN,NaN,NaN,30.614,1584.936,NaN,NaN,NaN,0 days 01:20:11.399000,0 days 01:29:06.624000,NaT,NaT,NaT,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,SOFT,NaN,NaN,NaN


In [17]:
for col, dtype in laps_pit_stops_flat.dtypes.items():
    if dtype == 'timedelta64[ns]':
        laps_pit_stops_flat[col] = laps_pit_stops_flat[col].dt.total_seconds().astype(float)

laps_pit_stops_flat

,Driver,DriverNumber,num_pit_stops,pit_1_lap,pit_2_lap,pit_3_lap,pit_4_lap,pit_5_lap,pit_1_duration,pit_2_duration,pit_3_duration,pit_4_duration,pit_5_duration,pit_1_intime,pit_2_intime,pit_3_intime,pit_4_intime,pit_5_intime,pit_1_fresh,pit_2_fresh,pit_3_fresh,pit_4_fresh,pit_5_fresh,pit_1_tyre_before,pit_2_tyre_before,pit_3_tyre_before,pit_4_tyre_before,pit_5_tyre_before,pit_1_tyre_after,pit_2_tyre_after,pit_3_tyre_after,pit_4_tyre_after,pit_5_tyre_after
0,GAS,10,4,14.0,26.0,32.0,33.0,NaN,31.053,32.291,1453.525,56.077,NaN,3610.679,4725.618,5358.556,6910.239,NaN,True,True,True,False,NaN,WET,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN
1,PER,11,2,28.0,33.0,NaN,NaN,NaN,44.676,1581.172,NaN,NaN,NaN,4798.896,5351.663,NaN,NaN,NaN,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,SOFT,NaN,NaN,NaN
2,ALO,14,3,28.0,32.0,33.0,NaN,NaN,30.984,1488.331,57.588,NaN,NaN,4878.812,5317.450,6905.161,NaN,NaN,True,True,False,NaN,NaN,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN,NaN
3,LEC,16,2,28.0,33.0,NaN,NaN,NaN,30.958,1609.473,NaN,NaN,NaN,4784.152,5308.896,NaN,NaN,NaN,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,MEDIUM,NaN,NaN,NaN
4,STR,18,3,27.0,32.0,33.0,NaN,NaN,31.137,1492.073,65.406,NaN,NaN,4756.421,5294.191,6877.429,NaN,NaN,True,True,False,NaN,NaN,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN,NaN
5,TSU,22,3,25.0,32.0,33.0,NaN,NaN,30.696,1491.687,63.320,NaN,NaN,4599.754,5304.596,6893.306,NaN,NaN,True,True,True,NaN,NaN,INTERMEDIATE,MEDIUM,SOFT,NaN,NaN,MEDIUM,SOFT,SOFT,NaN,NaN
6,RIC,3,2,27.0,33.0,NaN,NaN,NaN,34.285,1575.137,NaN,NaN,NaN,4737.991,5366.051,NaN,NaN,NaN,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,SOFT,NaN,NaN,NaN
7,OCO,31,5,1.0,27.0,31.0,32.0,33.0,30.809,30.642,30.678,1478.975,56.736,2116.950,4782.796,5183.677,5328.344,6907.285,True,True,True,False,False,WET,INTERMEDIATE,MEDIUM,SOFT,MEDIUM,NaN,MEDIUM,SOFT,MEDIUM,MEDIUM
8,VER,33,2,27.0,33.0,NaN,NaN,NaN,30.543,1621.405,NaN,NaN,NaN,4680.076,5292.328,NaN,NaN,NaN,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,MEDIUM,NaN,NaN,NaN
9,NOR,4,2,28.0,33.0,NaN,NaN,NaN,30.614,1584.936,NaN,NaN,NaN,4811.399,5346.624,NaN,NaN,NaN,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,SOFT,NaN,NaN,NaN


## Weather

In [27]:
laps_pit_stops_weather = pd.merge_asof(left=laps_pit_stops_flat.sort_values('pit_1_intime'), right=weather_data, left_on='pit_1_intime', right_on='Time', direction='backward')

In [31]:
laps_pit_stops_weather

,Driver,DriverNumber,num_pit_stops,pit_1_lap,pit_2_lap,pit_3_lap,pit_4_lap,pit_5_lap,pit_1_duration,pit_2_duration,pit_3_duration,pit_4_duration,pit_5_duration,pit_1_intime,pit_2_intime,pit_3_intime,pit_4_intime,pit_5_intime,pit_1_fresh,pit_2_fresh,pit_3_fresh,pit_4_fresh,pit_5_fresh,pit_1_tyre_before,pit_2_tyre_before,pit_3_tyre_before,pit_4_tyre_before,pit_5_tyre_before,pit_1_tyre_after,pit_2_tyre_after,pit_3_tyre_after,pit_4_tyre_after,pit_5_tyre_after,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,rain_start,rain_stop,rain_start_time,rain_stop_time
0,OCO,31,5,1.0,27.0,31.0,32.0,33.0,30.809,30.642,30.678,1478.975,56.736,2116.950,4782.796,5183.677,5328.344,6907.285,True,True,True,False,False,WET,INTERMEDIATE,MEDIUM,SOFT,MEDIUM,NaN,MEDIUM,SOFT,MEDIUM,MEDIUM,2081.415,9.6,75.4,1011.5,True,17.3,203,0.2,False,False,NaN,NaN
1,VET,5,5,3.0,20.0,22.0,32.0,33.0,32.300,31.490,39.795,1452.656,52.378,2410.348,4151.436,4372.911,5361.592,6915.581,True,True,True,False,True,INTERMEDIATE,INTERMEDIATE,MEDIUM,MEDIUM,SOFT,NaN,MEDIUM,MEDIUM,SOFT,SOFT,2381.413,9.6,76.8,1011.5,True,16.5,91,0.3,False,False,NaN,NaN
2,MSC,47,4,5.0,21.0,31.0,32.0,NaN,51.024,32.524,1521.337,51.216,NaN,2712.244,4284.543,5300.908,6917.270,NaN,True,True,False,True,NaN,WET,INTERMEDIATE,SOFT,MEDIUM,NaN,NaN,SOFT,MEDIUM,MEDIUM,NaN,2681.414,9.7,74.7,1011.5,True,16.3,95,0.2,False,False,NaN,NaN
3,MAZ,9,4,12.0,23.0,31.0,32.0,NaN,31.111,31.487,1486.087,49.738,NaN,3440.446,4520.373,5338.423,6922.414,NaN,True,True,False,True,NaN,WET,INTERMEDIATE,SOFT,MEDIUM,NaN,NaN,SOFT,MEDIUM,MEDIUM,NaN,3401.415,10.2,68.2,1011.4,True,14.9,91,0.9,False,False,NaN,NaN
4,GAS,10,4,14.0,26.0,32.0,33.0,NaN,31.053,32.291,1453.525,56.077,NaN,3610.679,4725.618,5358.556,6910.239,NaN,True,True,True,False,NaN,WET,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN,3581.414,10.3,68.5,1011.3,True,14.3,82,0.8,False,False,NaN,NaN
5,TSU,22,3,25.0,32.0,33.0,NaN,NaN,30.696,1491.687,63.320,NaN,NaN,4599.754,5304.596,6893.306,NaN,NaN,True,True,True,NaN,NaN,INTERMEDIATE,MEDIUM,SOFT,NaN,NaN,MEDIUM,SOFT,SOFT,NaN,NaN,4541.409,10.9,68.6,1011.3,False,14.7,92,1.0,False,False,NaN,3761.415
6,RUS,63,1,26.0,NaN,NaN,NaN,NaN,29.988,NaN,NaN,NaN,NaN,4674.198,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN,INTERMEDIATE,NaN,NaN,NaN,NaN,MEDIUM,NaN,NaN,NaN,NaN,4661.416,11.0,67.7,1011.3,False,15.0,95,1.0,False,False,NaN,3761.415
7,RAI,7,3,26.0,32.0,33.0,NaN,NaN,30.324,1492.955,63.170,NaN,NaN,4675.376,5296.865,6883.098,NaN,NaN,True,True,False,NaN,NaN,INTERMEDIATE,MEDIUM,MEDIUM,NaN,NaN,MEDIUM,MEDIUM,MEDIUM,NaN,NaN,4661.416,11.0,67.7,1011.3,False,15.0,95,1.0,False,False,NaN,3761.415
8,VER,33,2,27.0,33.0,NaN,NaN,NaN,30.543,1621.405,NaN,NaN,NaN,4680.076,5292.328,NaN,NaN,NaN,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,MEDIUM,NaN,NaN,NaN,4661.416,11.0,67.7,1011.3,False,15.0,95,1.0,False,False,NaN,3761.415
9,SAI,55,2,27.0,33.0,NaN,NaN,NaN,30.852,1582.515,NaN,NaN,NaN,4732.113,5355.151,NaN,NaN,NaN,True,True,NaN,NaN,NaN,INTERMEDIATE,MEDIUM,NaN,NaN,NaN,MEDIUM,MEDIUM,NaN,NaN,NaN,4721.412,11.1,67.6,1011.2,False,15.2,102,1.0,False,False,NaN,3761.415


## Putting it Together

In [7]:
def transform_weather(weather_data: pd.DataFrame):
    # pass in race.weather_data
    weather_data['rain_start'] = (weather_data['Rainfall'] == True) & (weather_data['Rainfall'].shift(1) == False)
    weather_data['rain_stop'] = (weather_data['Rainfall'].shift(1) == True) & (weather_data['Rainfall'] == False)
    weather_data['rain_start_time'] = weather_data['Time'].where(weather_data['rain_start']) 
    weather_data['rain_stop_time'] = weather_data['Time'].where(weather_data['rain_stop'])

    weather_data['rain_start_time'] = weather_data['rain_start_time'].dt.total_seconds().astype(float)
    weather_data['rain_stop_time'] = weather_data['rain_stop_time'].dt.total_seconds().astype(float)

    weather_data['rain_start_time'] = weather_data['rain_start_time'].ffill()
    weather_data['rain_stop_time'] = weather_data['rain_stop_time'].ffill()

    weather_data['Time'] = weather_data['Time'].dt.total_seconds().astype(float)


    weather_data['max_temp'] = weather_data['TrackTemp'].max()
    weather_data['min_temp'] = weather_data['TrackTemp'].min()
    weather_data['avg_temp'] = weather_data['TrackTemp'].mean()
    weather_data['std_temp'] = weather_data['TrackTemp'].std()

    return weather_data

In [8]:
def transform_circuit_info(circuit_info: pd.DataFrame):
    corners = circuit_info.corners[['Angle', 'Distance']]
    return corners

In [10]:
def transform_start_end_positions(results: pd.DataFrame):
    return pd.concat([results['Position'],results['GridPosition']], axis=1).rename(columns={'Position': 'target', 'GridPosition': 'starting_position'}).astype({'starting_position': 'int', 'target': 'int'})

In [11]:
def transform_laps_pit_stops(laps: pd.DataFrame):
    laps = (
        laps[['Driver', 'DriverNumber', 'LapNumber', 'PitInTime', 'PitOutTime', 'FreshTyre', 'TyreLife', 'Compound', 'TrackStatus']]
        # calcualte pit stop duration
        .assign(
            pit_duration=lambda x: (
                x.PitOutTime.shift(-1) - x.PitInTime
            ).dt.total_seconds()
        )
        # the lap at which the pit stop was made
        .assign(
            pit_stop=lambda x: np.where(
                x.pit_duration > 0, x.LapNumber.astype(int), None
            )
        )
        # the strategy after the pit stop
        .assign(
            strategy_after_pit=lambda x: np.where(
                x.shift(-1).Compound.isin(["SOFT"]),
                "SOFT",
                np.where(
                    x.shift(-1).Compound.isin(["MEDIUM"]),
                    "MEDIUM",
                    np.where(x.shift(-1).Compound.isin(["HARD"]), "HARD", None),
                ),
            )
        )
        # rename the compound column to strategy before the pit stop
        .rename(columns={"Compound": "strategy_before_pit"})
        # filter only for laps with pit stops
        .query("pit_stop.notnull()")[
            [
                "Driver",
                "DriverNumber",
                "LapNumber",
                "PitInTime",
                "FreshTyre",
                "TyreLife",
                "TrackStatus",
                "pit_duration",
                "pit_stop",
                "strategy_after_pit",
                "strategy_before_pit",
            ]
        ]
    )

    pit_stop_counts = laps.groupby('Driver')['pit_stop'].nunique().reset_index().rename(columns={'pit_stop': 'num_pit_stops'})

    laps_pit_stops = pd.merge(left=laps, right=pit_stop_counts, on='Driver', how='left')

    # Flatten/pivot the above such that we have one row per driver, and columns for each pit stop (lap number, duration, strategy before and after the pit stop)
    laps_pit_stops['stop_num'] = laps_pit_stops.groupby('Driver').cumcount() + 1
    per_stop_cols = {
            "LapNumber": "lap",
            "pit_duration": "duration",
            "PitInTime": "intime",
            "FreshTyre": "fresh",
            "strategy_before_pit": "tyre_before",
            "strategy_after_pit": "tyre_after",
        }

    pivoted = laps_pit_stops.groupby('Driver').first()[["DriverNumber", "num_pit_stops"]].reset_index()


    for src_col, label in per_stop_cols.items():
        stop_pivot = (
                laps_pit_stops.pivot_table(index='Driver', columns="stop_num", values=src_col, aggfunc="first")
            )
        stop_pivot.columns = [f"pit_{int(n)}_{label}" for n in stop_pivot.columns]
        pivoted = pivoted.merge(stop_pivot.reset_index(), on='Driver', how="left")


    laps_pit_stops_flat = pivoted.sort_values('DriverNumber').reset_index(drop=True)
    for col, dtype in laps_pit_stops_flat.dtypes.items():
        if dtype == 'timedelta64[ns]':
            laps_pit_stops_flat[col] = laps_pit_stops_flat[col].dt.total_seconds().astype(float)


    return laps_pit_stops_flat

In [ ]:
def merge_laps_with_weather(laps, weather_data):
    return pd.merge_asof(
        left=laps.sort_values('pit_1_intime'),
        right=weather_data,
        left_on='pit_1_intime',
        right_on='Time',
        direction='backward'
    )

In [18]:
for year in [2021]:
    countries = list(fastf1.get_event_schedule(year).Location.unique())
    for country in ['Bahrain']:#countries:
        race = fastf1.get_session(year, country, "R")
        race.load()
        weather_data = transform_weather(race.weather_data)
        circuit_info = transform_circuit_info(race.get_circuit_info())
        start_end_positions = transform_start_end_positions(race.results)
        laps_pit_stops = transform_laps_pit_stops(race.laps)  
        df = merge_laps_with_weather(laps_pit_stops, weather_data)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '11'
core        WARNING 	Driver 44 completed the race distance 00:00.067000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '77', '4', '11', '16', '3', '55', '22', '18', '7', '99', '31', '63', '5', '47', '10', '6', '14', '9']


NameError: name 'np' is not defined